In [3]:
import polars as pl

pl.read_ndjson("downloaded/geometry_complete.jsonl").head(2)

category,subcategory,question,answer,reasoning,source
str,str,str,f64,str,str
"""Geometry""","""geometry""","""the perimeter of an equilatera…",5.0,"""n0 = 60.0 n1 = 45.0 t0 = n0 / …","""MathQA_Geometry"""
"""Geometry""","""geometry""","""a small table has a length of …",16.0,"""import math n0 = 12.0 n1 = 4.0…","""MathQA_Geometry"""


In [6]:
pl.read_ndjson("downloaded/arithmetic_1000.jsonl").head(2)

category,subcategory,question,answer,reasoning,source
str,str,str,f64,null,str
"""Arithmetic""","""arithmetic_mixed""","""38×42=""",1596.0,null,"""Math-401"""
"""Arithmetic""","""arithmetic_mixed""",""" ( 11 * 17) + 56=""",243.0,null,"""Math-401"""


In [10]:
import json
with open("downloaded/wordProblems_complete.json", "r", encoding='utf-8') as f:
    df = pl.from_dicts(json.load(f)).head(2)

df

category,subcategory,question,answer,reasoning,source
str,str,str,f64,str,str
"""Word Problems""","""add_sub""","""Joan found 70 seashells on the…",43.0,"""X = 70 - 27""","""AddSub"""
"""Word Problems""","""add_sub""","""There were 28 bales of hay in …",26.0,"""X = 54 - 28""","""AddSub"""


In [12]:
from tinyreasoner import chat_api

def _format_answer(answer):
    if isinstance(answer, float):
        if answer % 1 == 0: return int(answer)

    if isinstance(answer, int):
        return answer

    try:
        answer = float(answer)
        return _format_answer(answer)
    except Exception:
        return answer

def load():
    samples = []

    df = pl.read_ndjson("downloaded/geometry_complete.jsonl")
    for row in df.iter_rows(named=True):

        assert row["reasoning"]

        samples.append([
            chat_api.UserMessage(row["question"]),
            chat_api.AssistantTurn(chat_api.Text(str(_format_answer(row["answer"]))))
        ])

        samples.append([
            chat_api.UserMessage(f'{row["question"]}. Write python code to solve this and the output.'),
            chat_api.AssistantTurn(chat_api.Text(f'```py\n{row["reasoning"]}\n```\nAnswer: {_format_answer(row["answer"])}'))
        ])

    df = pl.read_ndjson("downloaded/arithmetic_1000.jsonl")
    for row in df.iter_rows(named=True):

        samples.append([
            chat_api.UserMessage(row["question"]),
            chat_api.AssistantTurn(chat_api.Text(str(_format_answer(row["answer"]))))
        ])

    with open("downloaded/wordProblems_complete.json", "r", encoding='utf-8') as f:
        df = pl.from_dicts(json.load(f)).head(2)

    for row in df.iter_rows(named=True):

        samples.append([
            chat_api.UserMessage(row["question"]),
            chat_api.AssistantTurn(chat_api.Text(str(_format_answer(row["answer"]))))
        ])


    return samples

load()[-1]

[UserMessage(type=user, content=There were 28 bales of hay in the barn . Tim stacked bales in the barn today . There are now 54 bales of hay in the barn . How many bales did he store in the barn ? ),
 AssistantTurn(type=assistant, blocks=[Text(type=text, text=26)])]